In [190]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from gradient_boost_impl import GradientBoostingRegressor as gbr
from sklearn.ensemble import GradientBoostingRegressor as base_gbr
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.metrics import mean_absolute_error, r2_score, root_mean_squared_error

In [191]:
df= pd.read_csv('../../../datasets/cardekho_data.csv')
df

,Car_Name,Year,Selling_Price,Present_Price,Kms_Driven,Fuel_Type,Seller_Type,Transmission,Owner
0,ritz,2014,3.35,5.59,27000,Petrol,Dealer,Manual,0
1,sx4,2013,4.75,9.54,43000,Diesel,Dealer,Manual,0
2,ciaz,2017,7.25,9.85,6900,Petrol,Dealer,Manual,0
3,wagon r,2011,2.85,4.15,5200,Petrol,Dealer,Manual,0
4,swift,2014,4.60,6.87,42450,Diesel,Dealer,Manual,0
...,...,...,...,...,...,...,...,...,...
296,city,2016,9.50,11.60,33988,Diesel,Dealer,Manual,0
297,brio,2015,4.00,5.90,60000,Petrol,Dealer,Manual,0
298,city,2009,3.35,11.00,87934,Petrol,Dealer,Manual,0
299,city,2017,11.50,12.50,9000,Diesel,Dealer,Manual,0


In [192]:
df.columns= df.columns.str.lower().str.replace(' ', '_')

In [193]:
#converting year into current time distance
df['years_passed'] = pd.to_datetime('today').year - df['year']

In [194]:
X= df.drop(['car_name', 'selling_price', 'year'], axis= 1)
y= df['selling_price']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size= 0.2, random_state= 192)

In [195]:
print('X_train shape: ', X_train.shape)
print('y_train shape: ', y_train.shape)
print('X_train shape: ', X_test.shape)
print('y_train shape: ', y_test.shape)

X_train shape:  (240, 7)
y_train shape:  (240,)
X_train shape:  (61, 7)
y_train shape:  (61,)


In [196]:
#separating column types for pipelines
num_cols= ['present_price', 'kms_driven', 'years_passed']
cat_cols= ['fuel_type', 'seller_type', 'transmission', 'owner']

In [197]:
# pipelines for preprocessor
num_pipeline= Pipeline(
    steps= [
        ('imputing', SimpleImputer(strategy= 'median')),
        ('scaling', StandardScaler())
    ]
)

cat_pipeline= Pipeline(
    steps= [
        ('imputing', SimpleImputer(strategy= 'most_frequent')),
        ('encoder', OneHotEncoder(handle_unknown= 'ignore'))
    ]
)

In [198]:
preprocessor= ColumnTransformer(
    transformers= [
        ('num_cols', num_pipeline, num_cols),
        ('cat_cols', cat_pipeline, cat_cols)
    ],
    remainder= 'passthrough',
    n_jobs= -1
)

In [199]:
X_train = preprocessor.fit_transform(X_train)
y_train= y_train.values

model= gbr(max_leaf_count= 8)
model.fit(X_train, y_train)

In [200]:
X_test= preprocessor.transform(X_test)
y_test= y_test.values

y_pred= model.predict(X_test)

In [201]:
r2 = r2_score(y_test, y_pred)
rmse= root_mean_squared_error(y_test, y_pred)
mae= mean_absolute_error(y_test, y_pred)

print(f'r2 score= {r2}')
print(f'root mean squared error= {rmse}')
print(f'mean absolute error= {mae}')

r2 score= 0.940555905398303
root mean squared error= 1.4889212024991985
mean absolute error= 0.6589434562450689


## Comparing with a baseline model

In [202]:
base_model= base_gbr()
base_model.fit(X_train, y_train)
y_preds_base= base_model.predict(X_test)

In [203]:
base_r2 = r2_score(y_test, y_preds_base)
base_rmse= root_mean_squared_error(y_test, y_preds_base)
base_mae= mean_absolute_error(y_test, y_preds_base)

print(f'base r2 score= {base_r2}')
print(f'base root mean squared error= {base_rmse}')
print(f'base mean absolute error= {base_mae}')

base r2 score= 0.9359916174377667
base root mean squared error= 1.5450259657921321
base mean absolute error= 0.7066827500847714
